# Vectorization, normalization, and missing values

The three preprocessing steps almost every project needs, with the mistakes each one invites.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 6 — The Universal Workflow of Machine Learning](../../../course-web-slides/ch06/index.html) &nbsp;·&nbsp; **Section:** 02 — Develop a model

---

## Everything becomes a float tensor

In [ ]:
import numpy as np

# Text -> integers -> multi-hot, as in chapters 4 and 5.
# Categorical -> one-hot.
# Images -> float in [0, 1].
# Everything else -> normalized floats.

categories = np.array(["red", "green", "blue", "green", "red"])
vocab = sorted(set(categories))
lookup = {v: i for i, v in enumerate(vocab)}
onehot = np.eye(len(vocab))[[lookup[c] for c in categories]]
print(vocab)
print(onehot)

> **Note** — One-hot, not integers. Encoding *red=0, green=1, blue=2* tells the model that green is between red and blue, which is a fact you invented.

## Normalization, and why it is not optional

In [ ]:
import keras
from keras import layers
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
# Two features on wildly different scales -- a common real-world shape.
X = np.column_stack([rng.normal(0, 1, 2000),
                     rng.normal(5000, 2000, 2000)]).astype("float32")
y = (X[:, 0] * 2 + (X[:, 1] - 5000) / 1000 > 0).astype("float32")

def run(data, label):
    keras.utils.set_random_seed(0)
    m = keras.Sequential([layers.Dense(16, activation="relu"),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer="rmsprop", loss="binary_crossentropy",
              metrics=["accuracy"])
    h = m.fit(data, y, epochs=20, batch_size=64, validation_split=.3, verbose=0)
    print(f"{label:14s} best val acc {max(h.history['val_accuracy']):.4f}")
    return h

raw = run(X, "raw")
Xn = (X - X.mean(axis=0)) / X.std(axis=0)
norm = run(Xn, "normalized")

plt.figure(figsize=(6.5, 4))
plt.plot(raw.history["val_accuracy"], label="raw features")
plt.plot(norm.history["val_accuracy"], label="normalized")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend()
plt.title("One feature 2000x larger than the other")
plt.show()

Large input values produce large gradient updates, which destabilise everything downstream. **Small, homogeneous values — roughly zero mean, unit variance** — is the rule, and it costs one line.

## Doing it inside the model

A `Normalization` layer keeps the statistics **with the model**, so they cannot be lost between training and serving. This is the shape of a whole class of production bugs, removed.

In [ ]:
norm_layer = layers.Normalization()
norm_layer.adapt(X[:1400])          # TRAINING data only

model = keras.Sequential([
    norm_layer,
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="rmsprop", loss="binary_crossentropy",
              metrics=["accuracy"])
model.fit(X[:1400], y[:1400], epochs=20, batch_size=64, verbose=0)
print("test:", model.evaluate(X[1400:], y[1400:], verbose=0))

# The statistics now travel with the saved model.
model.save("normalized_model.keras")
reloaded = keras.saving.load_model("normalized_model.keras")
print("after reload:", reloaded.evaluate(X[1400:], y[1400:], verbose=0))

> ⚠️ **`adapt()` is fitting.** Call it on the training split only. Calling it on everything is the leak from notebook 03 of chapter 5, wearing a Keras-shaped disguise.

## Missing values

In [ ]:
Xm = Xn.copy()
missing = rng.random(Xm.shape) < 0.1
Xm[missing] = np.nan
print(f"{missing.mean():.1%} of entries missing")

# 0 is a safe fill *after* normalization: it is the mean, and the network
# will learn to treat it as "no information" -- provided it sees such
# samples during training.
Xf = np.where(np.isnan(Xm), 0.0, Xm)
print("any NaN left:", bool(np.isnan(Xf).any()))

Two conditions make zero-filling work, and both are easy to break.

1. The data must be **normalized first**, so that 0 means *the average*, not *nothing*.
2. The network must **see missing values during training**. If they only appear at inference time, artificially remove some from the training data — otherwise the model has never met the pattern it is about to be given.

## The order these must happen in

```
split  ->  fit preprocessing on train  ->  apply to all splits
```

Not: preprocess, then split. Every leak in chapter 5's notebook 03 comes from getting this order wrong, and no framework will warn you.

---

## What to take away

- Everything becomes a float tensor; categories become one-hot, not integers.
- Normalize to roughly zero mean and unit variance — large inputs destabilise training.
- A `Normalization` layer keeps the statistics with the model, which removes a whole class of serving bugs.
- Split first, fit preprocessing on train, then apply. Always that order.